##### ### The University of Melbourne, School of Computing and Information Systems
# COMP30027 Machine Learning, 2026 Semester 1

## Assignment 1: Income Classification with Naïve Bayes


**Student ID(s):**     `1616714`


This iPython notebook is a template which you will use for your Assignment 1 submission.

**NOTE: YOU SHOULD ADD YOUR RESULTS, GRAPHS, AND FIGURES FROM YOUR OBSERVATIONS IN THIS FILE TO YOUR REPORT (the PDF file).** Results, figures, etc. which appear in this file but are NOT included in your report will not be marked.

**Adding proper comments to your code is MANDATORY. **

## 0 Setup

In [1]:
import pandas as pd
import numpy as np

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_fscore_support
)

import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import norm

# define features, trarget and output directory for diagrams
categorical_features = [ "workclass", "education", "marital-status", "occupation", "relationship", "race", "sex", "native-country"]
numerical_features = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
ALL_FEATURES = categorical_features + numerical_features
TARGET = "income"
OUT_DIR = 'Diagrams/'

# define smoothing parameter for Naive Bayes
ALPHA = 1.0

# load data into dataframes
train_df = pd.read_csv('Assignment1_data/adult_supervised_train.csv')
unlabelled_df = pd.read_csv("Assignment1_data/adult_unlabelled.csv")
test_df = pd.read_csv("Assignment1_data/adult_test.csv")



## 0.5. Helpers

In [2]:

def clean_data(df, concept=True):
    cleaned_df = df.copy()
    cleaned_df = cleaned_df.replace('?', np.nan)
    if "fnlwgt" in cleaned_df.columns:
        cleaned_df = cleaned_df.drop(columns=["fnlwgt"])
    cleaned_df = cleaned_df.dropna(subset=ALL_FEATURES)

    return cleaned_df

# Get class means and variances for the continuous features
def gaussian_params(df):
    params = {}
    for column in df['income'].unique():
        subset = df[df['income'] == column]
        params[column] = {
            'mean': subset[numerical_features].mean(),
            'var': subset[numerical_features].var()
        }
    return params

# Get class probabilities for the categorical features
def categorical_params(df):
    probs = {}
    for column in df['income'].unique():
        subset = df[df['income'] == column]
        probs[column] = {}
        for feature in categorical_features:
            value_counts = subset[feature].value_counts()
            total_count = len(subset)
            probs[column][feature] = (value_counts / total_count).to_dict()
    return probs
    
def gaussian_log_probs(x, mean, var):
    return -0.5 * np.log(2 * np.pi * var) - ((x - mean) ** 2) / (2 * var)


class MixedNaiveBayes:

    def __init__(self):
        # Set up the Gaussian and categorical models
        self.gnb = GaussianNB()
        self.cnb = CategoricalNB(alpha = ALPHA)

    def fit(self, X_cat, X_cont, y):
        # Fit both models on the training data
        self.gnb.fit(X_cont, y)
        self.cnb.fit(X_cat, y)
        self.classes = self.gnb.classes_

    def predict_log_proba(self, X_cont, X_cat):
        # Combine the log probabilities from both models
        log_prob_cont = self.gnb.predict_log_proba(X_cont)
        log_prob_cat = self.cnb.predict_log_proba(X_cat)
        log_class_prior = np.log(self.gnb.class_prior_)
        return log_prob_cat + log_prob_cont - log_class_prior # adjust for double counting priors

    def predict(self, X_cont, X_cat):
        log_probs = self.predict_log_proba(X_cont, X_cat)
        return self.classes[np.argmax(log_probs, axis=1)]

    def posterior_ratio(self, X_cont, X_cat):
        log_probs = self.predict_log_proba(X_cont, X_cat)
        
        return np.exp(log_probs[:, 1] - log_probs[:, 0])


def encode_features(df, fit_encoder=False):
    # Split the data into continuous and categorical arrays
    x_cont = df[numerical_features].values
    if fit_encoder:
        x_cat = encoder.fit_transform(df[categorical_features]).astype(int)
    else:
        x_cat = encoder.transform(df[categorical_features]).astype(int)
    x_cat = x_cat + 1 # shift by 1 so unseen categories (encoded as 0) are distinguishable from seen ones
    return x_cont, x_cat

def create_mixed_nb():
    # Create the mixed model with space for unseen categorical values
    min_categories = [len(cats) + 1 for cats in encoder.categories_]
    model_local = MixedNaiveBayes()
    model_local.cnb = CategoricalNB(alpha=ALPHA, min_categories=min_categories)
    return model_local

def fit_mixed_nb_on_df(df):
    # Fit a mixed Naive Bayes model to a dataframe
    model_local = create_mixed_nb()
    x_cont_local, x_cat_local = encode_features(df, fit_encoder=False)
    y_local = df[TARGET].values
    model_local.fit(x_cat_local, x_cont_local, y_local)
    return model_local

def stable_softmax(log_scores):
    # Turn log scores into probabilities in a stable way
    shifted = log_scores - np.max(log_scores, axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

def pseudo_label_unlabelled(base_model, unlabeled_df, threshold=0.90):
    # Pseudo-label the unlabelled data and keep confident rows only (above threshold)
    x_u_cont, x_u_cat = encode_features(unlabeled_df, fit_encoder=False)
    log_scores_u = base_model.predict_log_proba(x_u_cont, x_u_cat)
    probs_u = stable_softmax(log_scores_u)

    confidence = np.max(probs_u, axis=1)
    pseudo_index = np.argmax(probs_u, axis=1)
    pseudo_labels = base_model.classes[pseudo_index]

    keep_mask = confidence >= threshold
    selected = unlabeled_df.loc[keep_mask].copy()
    selected[TARGET] = pseudo_labels[keep_mask]

    return selected, confidence, probs_u

def iterative_label_propagation(train_df, unlabeled_df, threshold=0.90, split_fraction=0.5, random_state=1):
    """
    For label propagation, the steps are:
    1. Train on labelled data
    2. Label the first half of the unlabelled data
    3. Retrain
    4. Label the second half
    5. Train a final model on all selected rows
    """
    shuffled = unlabeled_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    split_index = int(len(shuffled) * split_fraction)
    first_half = shuffled.iloc[:split_index].copy()
    second_half = shuffled.iloc[split_index:].copy()

    stage1_model = fit_mixed_nb_on_df(train_df)
    selected_1, _, _ = pseudo_label_unlabelled(stage1_model, first_half, threshold=threshold)

    stage2_train_df = pd.concat([train_df, selected_1], ignore_index=True)
    stage2_model = fit_mixed_nb_on_df(stage2_train_df)

    selected_2, _, _ = pseudo_label_unlabelled(stage2_model, second_half, threshold=threshold)

    final_train_df = pd.concat([train_df, selected_1, selected_2], ignore_index=True)
    final_model = fit_mixed_nb_on_df(final_train_df)

    return {
        "model": final_model,
        "final_train_df": final_train_df,
        "selected_stage1": selected_1,
        "selected_stage2": selected_2,
        "n_selected_stage1": len(selected_1),
        "n_selected_stage2": len(selected_2),
        "n_selected_total": len(selected_1) + len(selected_2),
        "threshold": threshold,
    }


## 1. Supervised model training


In [3]:
# Load the data and clean it
cleaned_train_df = clean_data(train_df)
cleaned_unlabelled_df = clean_data(unlabelled_df)
cleaned_test_df = clean_data(test_df)


encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

# Split the training data into feature arrays and labels
x_cont_train, x_cat_train = encode_features(cleaned_train_df, fit_encoder=True)
y_train = cleaned_train_df[TARGET].values

model = create_mixed_nb()
model.fit(x_cat_train, x_cont_train, y_train)

# Get the class priors
class_priors = dict(zip(model.classes, model.gnb.class_prior_))


print("Q1.1 Class Priors\n")
for class_label, prior_probability in class_priors.items():
    print(f"Class {class_label}: {prior_probability:.4f}")

print("\nQ1.2 Continuous Feature analysis")

# Get the Gaussian means and variances
means = model.gnb.theta_
vars_ = model.gnb.var_

feature_scores = []

# Score the continuous features by class separation
for i, col in enumerate(numerical_features):
    mu0, mu1 = means[0][i], means[1][i]
    var0, var1 = vars_[0][i], vars_[1][i]

    std1, std2 = np.sqrt(var0), np.sqrt(var1)
    
    separation = abs(mu1 - mu0) / np.sqrt((var0 + var1) / 2)
    feature_scores.append((col, mu0, mu1, std1, std2, separation))

# Sort by separation
feature_scores.sort(key=lambda x: x[5], reverse=True)

print("\nFeature | Mean(≤50K) | Mean(>50K) | Std Dev(≤50K) | Std Dev(>50K) | Separation")
for f in feature_scores:
    print(f"{f[0]:20} {f[1]:10.2f} {f[2]:10.2f} {f[3]:10.4f} {f[4]:10.4f} {f[5]:10.4f}")


print("\nQ1.3 Categorical Feature analysis (R values)")

feature_log_probs = model.cnb.feature_log_prob_
ratios = []

# Score each categorical value by its class ratio
for i, col in enumerate(categorical_features):
    categories = encoder.categories_[i]

    for j, val in enumerate(categories):
        # Shift by 1 because 0 is reserved for unseen values
        prob_idx = j + 1
        log_p1 = feature_log_probs[i][1][prob_idx]
        log_p0 = feature_log_probs[i][0][prob_idx]

        ratio = np.exp(log_p1 - log_p0)
        ratios.append((col, val, ratio))

ratios_sorted = sorted(ratios, key=lambda x: x[2], reverse=True)

print("\nTop 5 for >50K:")
for r in ratios_sorted[:5]:
    print(f"{r[0]} | {r[1]} | {r[2]:.4f}")

print("\nTop 5 for ≤50K:")
for r in ratios_sorted[-5:]:
    print(f"{r[0]} | {r[1]} | {r[2]:.4f}")

# Q1.4: Visualizations
class_labels = model.classes
class_counts = np.array([(y_train == cls).sum() for cls in class_labels])
class_priors_plot = np.array([class_priors[cls] for cls in class_labels])

means = model.gnb.theta_
stds = np.sqrt(model.gnb.var_)

ratios_df = pd.DataFrame(ratios_sorted, columns=["Feature", "Value", "R"])
top_high = ratios_df.nlargest(5, "R").copy()
top_low = ratios_df.nsmallest(5, "R").copy()
top_low["R"] = 1 / top_low["R"]

# Figure 1: class distribution
fig, ax = plt.subplots(figsize=(5, 3.5))
bar_labels = ["≤50K", ">50K"]
bars = ax.bar(
    bar_labels,
    class_counts,
    color=["#4C72B0", "#DD8452"],
    width=0.5,
    edgecolor="white",
    linewidth=1.5
)
ax.set_title("Class Distribution — Supervised Training Set", fontweight="bold")
ax.set_ylabel("Count")

for bar, count, prior in zip(bars, class_counts, class_priors_plot):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 40,
        f"{int(count):,}\n({prior:.1%})",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

ax.set_ylim(0, class_counts.max() * 1.18)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}q1_class_distribution.png", bbox_inches="tight")
plt.close()

# Figure 2: continuous feature distributions
fig, axes = plt.subplots(1, len(numerical_features), figsize=(18, 3.8), sharey=False)
colors_cls = ["blue", "orange"]
label_names = ["≤50K", ">50K"]

for i, feat in enumerate(numerical_features):
    ax = axes[i]
    fi = numerical_features.index(feat)

    for c in [0, 1]:
        mu = means[c, fi]
        sigma = stds[c, fi]
        x = np.linspace(mu - 3.5 * sigma, mu + 3.5 * sigma, 400)
        ax.plot(
            x,
            norm.pdf(x, mu, sigma),
            color=colors_cls[c],
            label=label_names[c],
            linewidth=2
        )
        ax.axvline(mu, color=colors_cls[c], linestyle="--", alpha=0.4, linewidth=1)

    ax.set_title(f"{feat}\n", fontweight="bold", fontsize=9.5)
    ax.set_xlabel("Value", fontsize=8)
    if i == 0:
        ax.set_ylabel("Density", fontsize=8)
    ax.legend(fontsize=7.5)
    ax.tick_params(labelsize=7.5)

plt.suptitle(
    "Gaussian Distributions per Class — Continuous Features",
    fontweight="bold",
    fontsize=12,
    y=1.01
)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}q1_gaussian_distributions.png", bbox_inches="tight")
plt.close()

# Figure 3: categorical predictor ratios
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, data, title, color in [
    (axes[0], top_high, "Top 5 Predictors of >50K", "#DD8452"),
    (axes[1], top_low, "Top 5 Predictors of ≤50K", "#4C72B0"),
]:
    labels_ = [f"{r['Feature']}\n= {r['Value']}" for _, r in data.iterrows()]
    vals = data["R"].values

    h_bars = ax.barh(labels_, vals, color=color, edgecolor="white", linewidth=1)

    for bar, v in zip(h_bars, vals):
        ax.text(
            v + max(vals) * 0.02,
            bar.get_y() + bar.get_height() / 2,
            f"{v:.2f}",
            va="center",
            fontsize=8.5
        )

    ax.set_xlabel("R →")
    ax.set_title(title, fontweight="bold")
    ax.invert_yaxis()

plt.suptitle(
    "Most Predictive Categorical Feature Values",
    fontweight="bold",
    fontsize=12
)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}q1_categorical_ratios.png", bbox_inches="tight")
plt.close()


Q1.1 Class Priors

Class <=50K: 0.7541
Class >50K: 0.2459

Q1.2 Continuous Feature analysis

Feature | Mean(≤50K) | Mean(>50K) | Std Dev(≤50K) | Std Dev(>50K) | Separation
education-num              9.63      11.59     2.4478     2.3650     0.8166
age                       37.05      43.94    13.7119    10.3027     0.5679
hours-per-week            39.43      45.64    11.9108    10.3961     0.5558
capital-gain             157.67    3607.15  1017.8632 13616.6230     0.3573
capital-loss              55.97     202.36   316.0201   603.9910     0.3037

Q1.3 Categorical Feature analysis (R values)

Top 5 for >50K:
education | Prof-school | 7.9572
education | Doctorate | 7.1341
marital-status | Married-AF-spouse | 4.2874
education | Masters | 3.5545
native-country | Taiwan | 3.3825

Top 5 for ≤50K:
education | 9th | 0.1508
relationship | Other-relative | 0.1445
occupation | Other-service | 0.1354
occupation | Priv-house-serv | 0.0838
relationship | Own-child | 0.0522


## 2. Supervised model evaluation

In [4]:
# Split the test data into features and labels
x_test_cont, x_test_cat = encode_features(cleaned_test_df, fit_encoder=False)
y_test = cleaned_test_df[TARGET].values

print("Model performance on test set:\n")
y_predictions = model.predict(x_test_cont, x_test_cat)
print(f"Accuracy: {accuracy_score(y_test, y_predictions):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_predictions, digits=4))
print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_predictions)
print(cm)

# Check how many test rows contain unseen categorical values
print("\nUnseen Category analysis")
x_test_cat_raw = encoder.transform(cleaned_test_df[categorical_features]).astype(int)
unseen_category = (x_test_cat_raw == -1) # True where the value was unseen in training
rows_with_unseen = np.any(unseen_category, axis=1)
print(f"Instances with unseen categories: {np.sum(rows_with_unseen)}")

print("\nConfidence analysis for test set predictions:\n")

# Get the combined log scores for each class
log_scores = model.predict_log_proba(x_test_cont, x_test_cat)

# Turn log scores into probabilities
y_proba = stable_softmax(log_scores)

idx_low = np.where(model.classes == "<=50K")[0][0]
idx_high = np.where(model.classes == ">50K")[0][0]

# Get the posterior ratio R
R = np.exp(log_scores[:, idx_high] - log_scores[:, idx_low])

results = cleaned_test_df.copy()
results["pred"] = y_predictions
results["true"] = y_test
results["R"] = R
results["P>50K"] = y_proba[:, idx_high]

# Key columns to print for the confidence examples
key_cols = [
    "age", "education", "occupation", "marital-status",
    "hours-per-week", "capital-gain", "sex", "pred", "true", "R", "P>50K"
]

high_label = model.classes[idx_high]
low_label = model.classes[idx_low]

# Split the test rows by prediction and confidence
hc_pos = results[results["pred"] == high_label].nlargest(5, "R")
hc_neg = results[results["pred"] == low_label].nsmallest(5, "R")
boundary = results.iloc[np.argsort(np.abs(results["R"] - 1.0))[:5]]

print("\n--- High confidence >50K ---")
print(hc_pos[key_cols].head(3).to_string(index=False))

print("\n--- High confidence <=50K ---")
print(hc_neg[key_cols].head(3).to_string(index=False))

print("\n--- Near boundary (R ~= 1) ---")
print(boundary[key_cols].head(3).to_string(index=False))

# Save the confusion matrix figure
fig, ax = plt.subplots(figsize=(5, 4))
cm_q1 = confusion_matrix(y_test, y_predictions)
ConfusionMatrixDisplay(cm_q1, display_labels=['≤50K','>50K']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Supervised Model (Q1)', fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}q2_confusion_matrix.png", bbox_inches='tight')
plt.close()

log_R = log_scores[:, idx_high] - log_scores[:, idx_low]
log_R_q1 = np.clip(log_R / np.log(10), -12, 12) 

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for pred_label, panel_label, color, ax in [
    (low_label, "≤50K", "blue", axes[0]),
    (high_label, ">50K", "orange", axes[1]),
]:
    mask = (y_predictions == pred_label)  
    ax.hist(log_R_q1[mask], bins=60, color=color, alpha=0.85, edgecolor="white", linewidth=0.4)
    ax.axvline(0, color="black", linestyle="--", linewidth=1.2, label="R=1 boundary")
    ax.set_title(f"Confidence — Predicted {panel_label}", fontweight="bold")
    ax.set_xlabel("log10(R)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

plt.suptitle("Posterior Ratio Distribution — Supervised Model", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}q2_confidence_distribution.png", bbox_inches="tight")
plt.close()



Model performance on test set:

Accuracy: 0.8306
Classification Report:
              precision    recall  f1-score   support

       <=50K     0.8544    0.9329    0.8919     11303
        >50K     0.7239    0.5251    0.6087      3784

    accuracy                         0.8306     15087
   macro avg     0.7891    0.7290    0.7503     15087
weighted avg     0.8217    0.8306    0.8209     15087

Confusion Matrix:
[[10545   758]
 [ 1797  1987]]

Unseen Category analysis
Instances with unseen categories: 1

Confidence analysis for test set predictions:


--- High confidence >50K ---
 age   education      occupation     marital-status  hours-per-week  capital-gain  sex pred true   R  P>50K
  50 Prof-school  Prof-specialty Married-civ-spouse              80         99999 Male >50K >50K inf    1.0
  41   Doctorate  Prof-specialty Married-civ-spouse              70         99999 Male >50K >50K inf    1.0
  47     Masters Exec-managerial Married-civ-spouse              50         99999 Male >

/var/folders/y9/7889p4mx7v36hghy1gg8xzgh0000gn/T/ipykernel_36593/813641430.py:33: RuntimeWarning: overflow encountered in exp
  R = np.exp(log_scores[:, idx_high] - log_scores[:, idx_low])


## 3. Extending the model with semi-supervised training

In [5]:
# Split the labelled data into train and validation sets
label_train_df, label_val_df = train_test_split(
    cleaned_train_df,
    test_size=0.2,
    stratify=cleaned_train_df[TARGET],
    random_state=1
)


unlabelled_feature_df = cleaned_unlabelled_df.drop(columns=[TARGET], errors="ignore").copy()

# Train a baseline model on the labelled split
baseline_model = fit_mixed_nb_on_df(label_train_df)
x_val_cont, x_val_cat = encode_features(label_val_df, fit_encoder=False)
y_val = label_val_df[TARGET].values

baseline_val_pred = baseline_model.predict(x_val_cont, x_val_cat)
baseline_val_acc = accuracy_score(y_val, baseline_val_pred)


print("Option 1: Iterative Label Propagation")
print(f"Baseline validation accuracy: {baseline_val_acc:.4f}")

thresholds = [0.80, 0.90, 0.95, 0.99]
iterative_results = []

for threshold in thresholds:
    result = iterative_label_propagation(
        train_df=label_train_df,
        unlabeled_df=unlabelled_feature_df,
        threshold=threshold,
        split_fraction=0.5,
        random_state=1
    )

    candidate_model = result["model"]
    val_pred = candidate_model.predict(x_val_cont, x_val_cat)
    val_acc = accuracy_score(y_val, val_pred)

    iterative_results.append({
        "strategy": "iterative",
        "threshold": threshold,
        "n_pseudo_stage1": result["n_selected_stage1"],
        "n_pseudo_stage2": result["n_selected_stage2"],
        "n_pseudo_total": result["n_selected_total"],
        "val_accuracy": val_acc,
        "model": candidate_model,
        "augmented_train_df": result["final_train_df"],
    })

iterative_results_df = pd.DataFrame([
    {
        "strategy": row["strategy"],
        "threshold": row["threshold"],
        "n_pseudo_stage1": row["n_pseudo_stage1"],
        "n_pseudo_stage2": row["n_pseudo_stage2"],
        "n_pseudo_total": row["n_pseudo_total"],
        "val_accuracy": row["val_accuracy"],
    }
    for row in iterative_results
]).sort_values("val_accuracy", ascending=False)

print("\nIterative label propagation results:")
print(iterative_results_df.to_string(index=False))

best_iterative = max(iterative_results, key=lambda item: item["val_accuracy"])

best_strategy = "iterative"
best_threshold = float(best_iterative["threshold"])
best_validation_accuracy = float(best_iterative["val_accuracy"])

print("\nSelected strategy:")
print(f"Strategy = {best_strategy}")
print(f"Threshold = {best_threshold}")
print(f"Validation accuracy = {best_validation_accuracy:.4f}")

# Refit the final semi-supervised model on all labelled data
final_iterative = iterative_label_propagation(
    train_df=cleaned_train_df,
    unlabeled_df=unlabelled_feature_df,
    threshold=best_threshold,
    split_fraction=0.5,
    random_state=1
)

semi_supervised_model = final_iterative["model"]
final_augmented_train_df = final_iterative["final_train_df"]
n_selected_unlabelled = final_iterative["n_selected_total"]

model_ssm = semi_supervised_model

print("\nFinal semi-supervised model trained.")
print(f"Supervised training rows: {len(cleaned_train_df)}")
print(f"Pseudo-labelled rows added: {n_selected_unlabelled}")
print(f"Final training rows: {len(final_augmented_train_df)}")
print("\nFinal class distribution:")
print(final_augmented_train_df[TARGET].value_counts(normalize=True).sort_index())


Option 1: Iterative Label Propagation
Baseline validation accuracy: 0.8316

Iterative label propagation results:
 strategy  threshold  n_pseudo_stage1  n_pseudo_stage2  n_pseudo_total  val_accuracy
iterative       0.99             4829             5149            9978      0.817639
iterative       0.95             5792             6298           12090      0.808024
iterative       0.90             6306             6676           12982      0.805040
iterative       0.80             6781             6987           13768      0.804377

Selected strategy:
Strategy = iterative
Threshold = 0.99
Validation accuracy = 0.8176

Final semi-supervised model trained.
Supervised training rows: 15076
Pseudo-labelled rows added: 9974
Final training rows: 25050

Final class distribution:
income
<=50K    0.793812
>50K     0.206188
Name: proportion, dtype: float64


## 4. Supervised model evaluation

In [6]:
# Split the test data into features and labels
X_test_cont, X_test_cat = encode_features(cleaned_test_df, fit_encoder=False)
y_test = cleaned_test_df[TARGET].values

# Get predictions and scores for both models
y_predictions_q1 = model.predict(X_test_cont, X_test_cat)
log_scores_q1 = model.predict_log_proba(X_test_cont, X_test_cat)

# Get the class indices for the ratio
idx_low_q1 = np.where(model.classes == "<=50K")[0][0]
idx_high_q1 = np.where(model.classes == ">50K")[0][0]

# Get the raw posterior ratio values
log_R_q1 = log_scores_q1[:, idx_high_q1] - log_scores_q1[:, idx_low_q1]
ratios_q1 = np.exp(np.clip(log_R_q1, -700, 700))

print("\nQ1 Supervised")
print(classification_report(y_test, y_predictions_q1, target_names=["<=50K", ">50K"]))
print(f"Accuracy: {accuracy_score(y_test, y_predictions_q1):.4f}")

# Get predictions and scores for the semi-supervised model
y_predictions_ssl = model_ssm.predict(X_test_cont, X_test_cat)
log_scores_q3 = model_ssm.predict_log_proba(X_test_cont, X_test_cat)

# Get the class indices for the ratio
idx_low_q3 = np.where(model_ssm.classes == "<=50K")[0][0]
idx_high_q3 = np.where(model_ssm.classes == ">50K")[0][0]

# Get the raw posterior ratio values
log_R_q3 = log_scores_q3[:, idx_high_q3] - log_scores_q3[:, idx_low_q3]
ratios_q3 = np.exp(np.clip(log_R_q3, -700, 700))

print("\nQ3 Semi-Supervised")
print(classification_report(y_test, y_predictions_ssl, target_names=["<=50K", ">50K"]))
print(f"Accuracy: {accuracy_score(y_test, y_predictions_ssl):.4f}")

# Compare Gaussian parameters between Q1 (supervised) and Q3 (semi-supervised)
print("\nGaussian parameter shift (Q3 - Q1) for continuous features")

gnb_q1 = model.gnb
gnb_q3 = model_ssm.gnb

# Use the shared class labels
cls_low = "<=50K"
cls_high = ">50K"
idx_low_q1 = np.where(gnb_q1.classes_ == cls_low)[0][0]
idx_high_q1 = np.where(gnb_q1.classes_ == cls_high)[0][0]
idx_low_q3 = np.where(gnb_q3.classes_ == cls_low)[0][0]
idx_high_q3 = np.where(gnb_q3.classes_ == cls_high)[0][0]

rows = []
for j, feat in enumerate(numerical_features):
    mu_q1_low = gnb_q1.theta_[idx_low_q1, j]
    mu_q1_high = gnb_q1.theta_[idx_high_q1, j]
    mu_q3_low = gnb_q3.theta_[idx_low_q3, j]
    mu_q3_high = gnb_q3.theta_[idx_high_q3, j]

    std_q1_low = np.sqrt(gnb_q1.var_[idx_low_q1, j])
    std_q1_high = np.sqrt(gnb_q1.var_[idx_high_q1, j])
    std_q3_low = np.sqrt(gnb_q3.var_[idx_low_q3, j])
    std_q3_high = np.sqrt(gnb_q3.var_[idx_high_q3, j])

    rows.append({
        "feature": feat,
        "d_mean_<=50K": mu_q3_low - mu_q1_low,
        "d_mean_>50K": mu_q3_high - mu_q1_high,
        "d_std_<=50K": std_q3_low - std_q1_low,
        "d_std_>50K": std_q3_high - std_q1_high
    })

gaussian_shift_df = pd.DataFrame(rows)
print(gaussian_shift_df.to_string(index=False, float_format=lambda x: f"{x: .6f}"))


# Compare categorical feature values between Q1 and Q3
print("\nCategorical feature shift (Q3 vs Q1)")

feature_log_probs_q1 = model.cnb.feature_log_prob_
feature_log_probs_q3 = model_ssm.cnb.feature_log_prob_

cat_rows = []

for i, col in enumerate(categorical_features):
    categories = encoder.categories_[i]

    for j, val in enumerate(categories):
        # Shift by 1 because 0 is reserved for unseen values
        prob_idx = j + 1

        log_r_q1 = feature_log_probs_q1[i][1][prob_idx] - feature_log_probs_q1[i][0][prob_idx]
        log_r_q3 = feature_log_probs_q3[i][1][prob_idx] - feature_log_probs_q3[i][0][prob_idx]

        cat_rows.append({
            "feature": col,
            "value": val,
            "R_q1": np.exp(log_r_q1),
            "R_q3": np.exp(log_r_q3),
        })

cat_shift_df = pd.DataFrame(cat_rows)
cat_shift_df["delta_R"] = cat_shift_df["R_q3"] - cat_shift_df["R_q1"]
cat_shift_df["abs_change"] = cat_shift_df["delta_R"].abs()

print("\nTop 10 categorical values with the biggest change in R:")
print(
    cat_shift_df.sort_values("abs_change", ascending=False)
    .head(10)[["feature", "value", "R_q1", "R_q3", "delta_R"]]
    .to_string(index=False, float_format=lambda x: f"{x: .4f}")
)

# Save the confusion matrix figure
np.set_printoptions(suppress=True, precision=6) # Keep printed numbers readable
pd.set_option("display.float_format", "{:.6f}".format)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))

for ax, y_p, title in [
    (axes[0], y_predictions_q1, "Q1 — Supervised"),
    (axes[1], y_predictions_ssl, "Q3 — Semi-Supervised"),
]:
    cm = confusion_matrix(y_test, y_p)
    ConfusionMatrixDisplay(cm, display_labels=["≤50K", ">50K"]).plot(
        ax=ax, colorbar=False, cmap="Blues", values_format="d"
    )
    ax.set_title(title, fontweight="bold")

plt.suptitle("Confusion Matrices — Test Set Comparison", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}q4_confusion_matrices.png", bbox_inches="tight")
plt.close()

# Save the posterior ratio plot
log10_R_q1 = np.log10(np.clip(ratios_q1, 1e-12, 1e12))
log10_R_q3 = np.log10(np.clip(ratios_q3, 1e-12, 1e12))
bins = np.linspace(-12, 12, 100)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(
    log10_R_q1,
    bins=bins,
    alpha=0.55,
    label="Q1 Supervised",
    color="blue",
    edgecolor="white",
    linewidth=0.3,
)
ax.hist(
    log10_R_q3,
    bins=bins,
    alpha=0.55,
    label="Q3 Semi-Supervised",
    color="orange",
    edgecolor="white",
    linewidth=0.3,
)
ax.axvline(0, color="black", linestyle="--", linewidth=1.5, label="Boundary (R=1)")
ax.set_xlabel("log10(R) = log10[P(>50K|x) / P(≤50K|x)]")
ax.set_ylabel("Count")
ax.set_title("Posterior Ratio Distribution Shift — Q1 vs Q3", fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}q4_confidence_shift.png", bbox_inches="tight")
plt.close()




Q1 Supervised
              precision    recall  f1-score   support

       <=50K       0.85      0.93      0.89     11303
        >50K       0.72      0.53      0.61      3784

    accuracy                           0.83     15087
   macro avg       0.79      0.73      0.75     15087
weighted avg       0.82      0.83      0.82     15087

Accuracy: 0.8306

Q3 Semi-Supervised
              precision    recall  f1-score   support

       <=50K       0.84      0.93      0.88     11303
        >50K       0.69      0.49      0.57      3784

    accuracy                           0.82     15087
   macro avg       0.77      0.71      0.73     15087
weighted avg       0.81      0.82      0.81     15087

Accuracy: 0.8181

Gaussian parameter shift (Q3 - Q1) for continuous features
       feature  d_mean_<=50K  d_mean_>50K  d_std_<=50K   d_std_>50K
           age     -0.943334     0.119976     0.028489     0.314193
 education-num     -0.153721     0.044559    -0.013013     0.039757
  capital-gai